# 00. Generate synthetic data and inspect native measurements
This notebook generates data, checks structure, and inspects
native measurements, missingness, gaps and development fault scenarios.
Full-data checks are structural only. Final-test fault details remain hidden.
Statistical canonical EDA follows in notebook 01; model features follow in 02.
Passing consistency checks does not certify real-world realism.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation
from optical_anomaly.workflow import run_split

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
split = run_split(settings)
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [
    split.train_end,
    split.calibration_end,
    split.validation_end,
    split.test_end,
]
REPORT = RUN / "eda"
REPORT.mkdir(exist_ok=True)
print("Run:", RUN.resolve())
from optical_anomaly.generator import GeneratorConfig
from optical_anomaly.optics import validate_generated
from optical_anomaly.diagnostics import (
    missingness_report,
    development_faults,
    fault_contrasts,
)
from optical_anomaly.sources import SYNTHETIC_METRICS
from optical_anomaly.adapter import CANONICAL_METRICS

config = GeneratorConfig(**settings["generator"])

## A. Full-dataset inventory and structural qualification
No final-test fault details are displayed. Counts describe the dataset, not model performance.

In [ ]:
native = pd.read_parquet(RUN / "telemetry.parquet")
truth = pd.read_parquet(RUN / "ground_truth.parquet")
topology = pd.read_parquet(RUN / "topology.parquet")
report = validate_generated(native, truth, topology)
step = pd.Timedelta(minutes=config.interval_minutes)
expected_rows = int(config.days * 1440 / config.interval_minutes)
groups = native.groupby("device", sort=True)
checks = dict(report["checks"])
checks["timestamps_present"] = bool(native.time.notna().all())
checks["configured_entity_count"] = native.device.nunique() == config.entities
checks["configured_rows_per_entity"] = bool(groups.size().eq(expected_rows).all())
checks["ordered_fixed_cadence"] = all(
    group.time.diff().dropna().eq(step).all() and group.time.iloc[0] == start
    for _, group in groups
)
visible = truth.dropna(subset=["observable_onset_time"])
checks["observable_onset_inside_fault"] = bool(
    (
        visible.observable_onset_time.ge(visible.onset_time)
        & visible.observable_onset_time.lt(visible.end_time)
    ).all()
)
checks["unique_fault_ids"] = not truth.fault_id.duplicated().any()
checks = {name: bool(value) for name, value in checks.items()}
display(
    pd.Series(
        {
            "rows": len(native),
            "ONTs": native.device.nunique(),
            "measurements": len(SYNTHETIC_METRICS),
            "days": config.days,
            "interval_minutes": config.interval_minutes,
            "splitters": topology.splitter_id.nunique(),
            "ports": topology.pon_port_id.nunique(),
            "OLTs": topology.olt_id.nunique(),
        }
    )
)
display(pd.Series(checks, name="structural_pass"))
(REPORT / "structural_checks.json").write_text(json.dumps(checks, indent=2))
assert all(checks.values()), "Fix structural failures before adapting data"
# Drop final data from subsequent inspection.
native = native.loc[native.time < boundaries[2]].copy()
truth = truth.loc[truth.onset_time < boundaries[2]].copy()
healthy = native.loc[native.time < boundaries[0]].copy()

## B. Measurement dictionary and topology
OLT Tx and temperature are shared port observations repeated per ONT. Do not treat those rows as independent port measurements. FEC counts describe the preceding interval; its first row is intentionally missing.

In [ ]:
# Familiar display labels; canonical names and saved measurements stay precise.
labels = {source: source.replace("_", " ") for source in SYNTHETIC_METRICS}
for direction in ("downstream", "upstream"):
    for kind in ("corrected", "uncorrectable", "total"):
        labels[f"{direction}_fec_{kind}_codewords"] = (
            f"{direction.title()} FEC — "
            f"{'received' if kind == 'total' else kind} blocks"
        )
dictionary = pd.DataFrame(
    [
        {
            "display_name": labels[source],
            "source": source,
            "canonical": name,
            "unit": CANONICAL_METRICS[name].unit,
            "kind": CANONICAL_METRICS[name].kind,
            "meaning": CANONICAL_METRICS[name].description,
            "used_by_rx_baseline": name == "rx_power_dbm",
        }
        for source, name in SYNTHETIC_METRICS.items()
    ]
)
display(dictionary)
dictionary.to_csv(REPORT / "measurement_dictionary.csv", index=False)
display(topology.head())
display(
    topology.groupby(["olt_id", "pon_port_id"]).agg(
        ONTs=("entity_id", "size"), splitters=("splitter_id", "nunique")
    )
)

Tables and plots use familiar **FEC corrected blocks**, **FEC uncorrectable blocks**
and **FEC received blocks** labels. Both upstream and downstream are explicit. Here a block means a codeword; it does not mean
corrected bits or bytes. Original column names remain visible in the dictionary.
The synthetic total counts all received codewords. A vendor field called “total
FEC” may use a different denominator: verify its definition before mapping it.

## C. Counts, missingness, gaps and distributions — development only
Longest gaps below are missing polls on the verified fixed grid. Constant counters can be legitimate when no errors occur; constant optical power deserves inspection.

In [ ]:
coverage = missingness_report(native)
coverage.to_csv(REPORT / "development_missingness.csv", index=False)
labelled_coverage = coverage.assign(measurement=coverage.measurement.map(labels))
display(
    labelled_coverage.groupby("measurement").agg(
        observed=("observed", "sum"),
        mean_missing_fraction=("missing_fraction", "mean"),
        longest_gap_intervals=("longest_missing_intervals", "max"),
        minimum_unique_values=("unique_observed_values", "min"),
    )
)
display(coverage.sort_values("missing_fraction", ascending=False).head(20))
summary = native[list(SYNTHETIC_METRICS)].describe(percentiles=[0.01, 0.5, 0.99]).T
display(summary.rename(index=labels))
summary.to_csv(REPORT / "development_distributions.csv")

In [ ]:
entity = healthy.device.iloc[0]
view = healthy.loc[healthy.device.eq(entity)].set_index("time")
view[["rx_dbm", "upstream_rx_dbm"]].plot(
    figsize=(12, 3), ylabel="Rx (dBm)", title="Native healthy measurements"
)
plt.show()

## D. Generator-specific statistical qualification
Check the healthy path-loss daily amplitude, residual variance and correlation
against the declared generator settings. These are internal consistency screens,
not field-calibrated limits. Statistical-band failures require investigation rather
than silent retuning. Canonical EDA in 01 separately asks which daily/weekly patterns
predict held-out healthy observations.

In [ ]:
from optical_anomaly.generator import GeneratorConfig
from optical_anomaly.diagnostics import healthy_statistics

native_training = pd.read_parquet(
    RUN / "telemetry.parquet",
    filters=[("time", "<", split.train_end)],
    columns=["time", "device", "rx_dbm", "olt_tx_dbm"],
)
checks = healthy_statistics(native_training, GeneratorConfig(**settings["generator"]))
checks.to_csv(REPORT / "healthy_statistics.csv", index=False)
display(checks)
display(checks.filter(like="in_band").mean().rename("fraction_in_band"))

## E. Faults and dependencies — development only
Inspect duration diversity, subtle versus large changes, non-impacting faults and
warning opportunities. The opportunity column is an availability proxy, not recall.
Observable onset uses the simulator's known effect, not a field-observable truth.
BER is internal only and is not exported. FEC uses the preceding physical sample
with independent receiver offsets and interval dispersion (simulation assumptions).
Strong relationships are partly constructed; they are not independent validation.

In [ ]:
faults = development_faults(truth, native, boundaries[2], config.interval_minutes)
faults.to_csv(REPORT / "development_faults.csv", index=False)
display(
    faults.groupby("fault_type").agg(
        faults=("fault_id", "size"),
        impacts=("impact_time", "count"),
        opportunities=("warning_opportunity_proxy", "sum"),
        median_duration_hours=("duration_hours", "median"),
        median_warning_minutes=("warning_minutes", "median"),
    )
)
for fault in faults.groupby("fault_type").head(1).itertuples():
    sample = native.loc[
        native.device.eq(fault.entity_id)
        & native.time.between(
            fault.onset_time - pd.Timedelta(hours=12),
            fault.end_time + pd.Timedelta(hours=6),
        )
    ].set_index("time")
    ax = sample[["rx_dbm", "upstream_rx_dbm"]].plot(
        figsize=(11, 3),
        title=f"{fault.entity_id}: {fault.fault_type}",
        ylabel="Rx (dBm)",
    )
    for name in ["onset_time", "observable_onset_time", "impact_time", "end_time"]:
        when = getattr(fault, name)
        if pd.notna(when):
            ax.axvline(when, linestyle="--", label=name)
    ax.legend(fontsize=8)
    plt.show()
# One ONT avoids treating shared port rows as independent observations.
view = native.loc[native.device.eq(entity)].set_index("time")
ratios = pd.DataFrame(index=view.index)
for direction in ["downstream", "upstream"]:
    total = view[f"{direction}_fec_total_codewords"].where(lambda x: x > 0)
    for kind in ["corrected", "uncorrectable"]:
        ratios[f"{direction.title()} FEC {kind} fraction"] = (
            view[f"{direction}_fec_{kind}_codewords"] / total
        )
ratios.plot(figsize=(12, 3), ylabel="FEC fraction", title=entity)
plt.show()
display(
    view[["rx_dbm", "upstream_rx_dbm", "ont_tx_dbm", "olt_tx_dbm"]].corr(
        method="spearman"
    )
)
contrasts = fault_contrasts(native, faults)
contrasts.to_csv(REPORT / "development_fault_contrasts.csv", index=False)
display(contrasts)
print("Inspect large drops, low severity diversity, and fault-associated missingness.")

## F. Qualification decision and remaining limitations
**Must pass:** structural checks. **Investigate:** statistical-band failures,
long gaps, constant optical measurements, implausibly easy faults, and limited
healthy exposure. No single automatic score certifies synthetic realism.

Current limitations remain explicit:
- Fault timing is concentrated in scenario windows, with enriched fault prevalence.
- Missing polls are independent of severity; outage-related censoring is absent.
- No shared splitter/port faults or empirical hardware/receiver calibration.
- BER and FEC relationships are simulated; real vendor counter semantics may differ.
- Multiple seeds and parameter stress experiments are needed before robustness claims.
- Final-test details remain excluded from these development diagnostics.

Next run notebook 01 for canonical adaptation and statistical EDA.
Review `METHOD.md` for standards, assumptions and evidence. Reports are written to
`RUN/eda/`; the original telemetry and truth files are not modified.

The export contains optical power, temperature and FEC measurements. `impact_time`
is the third consecutive latent low-power reading, not the first reading of that
run and not verified service loss. Variance shifts have no asserted observable
onset; their detection delay uses physical onset separately. Receiver parameters
are independent of label thresholds. FEC remains an assumed receiver model; the
added dispersion is not evidence of realistic field counter behaviour.